In [1]:
import torchvision
import torch
import torchvision.transforms as transforms
import os
import matplotlib.pyplot as plt
import numpy as np

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models
# from torchinfo import summary
import numpy as np
import matplotlib.pyplot as plt
import os
import random
from torchvision import  transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import ImageFolder
from tqdm import tqdm
from PIL import Image
import warnings

In [3]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224), interpolation=Image.BICUBIC),
    transforms.RandomHorizontalFlip(p=0.2),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=2),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    # transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    # transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5493, 0.3938, 0.3555], std=[0.2206, 0.1684, 0.1597]),
    # transforms.RandomErasing(p=0.1, scale=(0.02, 0.1), ratio=(0.3, 3.3), value='random')
])


test_transforms = transforms.Compose([
    transforms.Resize((224, 224), interpolation=Image.BICUBIC),
    # transforms.CenterCrop((64, 64)),
    transforms.RandomHorizontalFlip(p=0.2),
    # transforms.RandomVerticalFlip(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5493, 0.3938, 0.3555], std=[0.2206, 0.1684, 0.1597])
])



In [4]:


def get_dataloaders(root_dir, batch_size, num_workers,test_fraction=0.8, validation_fraction=0.1):
    """
    Creates train, validation, and test DataLoaders from a structured dataset folder.

    :param root_dir: Path to the dataset folder containing train, val, and test subfolders.
    :param batch_size: Batch size for training/testing.
    :param num_workers: Number of workers for data loading.
    :param validation_fraction: Fraction of training data to use as validation (if val folder is not available).
    :return: train_loader, valid_loader, test_loader
    """
    # Load datasets from folder
    train_dataset = ImageFolder(root=f"{root_dir}/PetImages", transform=train_transforms)

    # If a validation folder exists, use it; otherwise, split from training data
    try:
        valid_dataset = ImageFolder(root=f"{root_dir}/val", transform=test_transforms)
        print("Images read from validation folder.")
    except:
        val_size = int(validation_fraction * len(train_dataset))
        train_size = len(train_dataset) - val_size
        train_dataset, valid_dataset = random_split(train_dataset, [train_size, val_size])

        test_size = int(test_fraction * len(train_dataset))
        train_size = len(train_dataset) - test_size
        train_dataset, test_dataset = random_split(train_dataset, [train_size, test_size])


    # DataLoaders
    train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    valid_loader = DataLoader(dataset=valid_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)

    return train_loader, valid_loader, test_loader

# # Example Usage



root_dir = r"/kaggle/input/datasets/bhavikjikadara/dog-and-cat-classification-dataset"  # Replace with your dataset path

train_loader, valid_loader, test_loader = get_dataloaders(root_dir, batch_size=4, num_workers=0)

# Checking the dataset
for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Label batch dimensions:', labels.shape)
    print('Class labels of 10 examples:', labels[:10])
    break



Image batch dimensions: torch.Size([4, 3, 224, 224])
Label batch dimensions: torch.Size([4])
Class labels of 10 examples: tensor([0, 0, 1, 1])


In [5]:
def set_device():
    if torch.cuda.is_available():
        dev = "cuda:0"
    else:
        dev = "cpu"

    return torch.device(dev)

In [6]:
def train_nn(model, train_loader, test_loader, criterion, optimizer, n_epochs):

    device = set_device()

    train_losses = []
    val_losses = []

    train_accs = []
    val_accs = []

    for epoch in range(n_epochs):

        print(f"\nEpoch {epoch+1}/{n_epochs}")

        model.train()

        running_loss = 0.0
        running_correct = 0
        total = 0

        for images, labels in train_loader:

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            running_loss += loss.item()
            running_correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = 100 * running_correct / total

        val_loss, val_acc = evaluate_model_on_test_set(
            model,
            test_loader,
            criterion
        )

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        train_accs.append(train_acc)
        val_accs.append(val_acc)

        print(f"Train Loss : {train_loss:.4f}")
        print(f"Train Acc  : {train_acc:.2f}%")
        print(f"Val Loss   : {val_loss:.4f}")
        print(f"Val Acc    : {val_acc:.2f}%")

    print("Finished Training")

    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "train_acc": train_accs,
        "val_acc": val_accs
    }

    return history

In [7]:
def evaluate_model_on_test_set(model, test_loader, criterion):
    model.eval()
    device = set_device()

    running_loss = 0.0
    running_correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            running_correct += (predicted == labels).sum().item()
            running_loss += loss.item()

    val_loss = running_loss / len(test_loader)
    val_acc = 100 * running_correct / total

    return val_loss, val_acc

In [8]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights

class ViTB16Model(nn.Module):
    def __init__(self, num_classes=2):  # Binary: 1 output neuron
        super(ViTB16Model, self).__init__()

        self.model = models.vit_b_16(weights=None)

        # Modify the head for binary classification
        num_features = self.model.heads.head.in_features
        self.model.heads.head = nn.Sequential(
            nn.Linear(num_features, 2),  # Single output for binary
            nn.Softmax()  
        )

    def forward(self, x):
        return self.model(x)

model = ViTB16Model(num_classes=2)  # Binary: 1 output neuron

In [9]:
import torchvision.models as models
import torch.nn as nn
import torch.optim as optim


device = set_device()
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(),lr=0.01,momentum=0.9,weight_decay=0.003)

In [ ]:
history = train_nn(
    model,
    train_loader,
    test_loader,
    loss_fn,
    optimizer,
    5
)


Epoch 1/5


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py:1776: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Train Loss : 0.8123
Train Acc  : 50.09%
Val Loss   : 0.8134
Val Acc    : 49.99%

Epoch 2/5
Train Loss : 0.8072
Train Acc  : 50.49%
Val Loss   : 0.8078
Val Acc    : 50.53%

Epoch 3/5
Train Loss : 0.8039
Train Acc  : 50.64%
Val Loss   : 0.8126
Val Acc    : 50.06%

Epoch 4/5


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(14,5))

# Loss Curve
plt.subplot(1,2,1)
plt.plot(epochs, history["train_loss"], label="Training Loss")
plt.plot(epochs, history["val_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Learning Curve (Loss)")
plt.legend()
plt.grid(True)

# Accuracy Curve
plt.subplot(1,2,2)
plt.plot(epochs, history["train_acc"], label="Training Accuracy")
plt.plot(epochs, history["val_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.title("Learning Curve (Accuracy)")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/efficientnet_weights.pth")


In [ ]:
model1 = model.load_state_dict(torch.load("/kaggle/working/efficientnet_weights.pth"))

In [ ]:
import torch

def test_model(model, test_loader, criterion):
    """
    Evaluate the model on the test dataset.

    Args:
        model: Trained PyTorch model.
        test_loader: DataLoader for the test dataset.
        criterion: Loss function.

    Returns:
        test_loss, test_accuracy, predictions, true_labels
    """

    device = set_device()
    model.to(device)
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    predictions = []
    true_labels = []

    with torch.no_grad():
        for images, labels in test_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, preds = torch.max(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            predictions.extend(preds.cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    test_loss = running_loss / len(test_loader)
    test_accuracy = 100 * correct / total

    print("=" * 50)
    print("Test Results")
    print("=" * 50)
    print(f"Test Loss     : {test_loss:.4f}")
    print(f"Test Accuracy : {test_accuracy:.2f}%")
    print("=" * 50)

    return test_loss, test_accuracy, predictions, true_labels

In [ ]:
test_loss, test_acc, y_pred, y_true = test_model(
    model,
    test_loader,
    loss_fn
)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=test_loader.dataset.classes
))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8,6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_loader.dataset.classes,
    yticklabels=test_loader.dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()